# Build Speaker-Level Sign Count Dataset

This notebook aggregates the utterance-level `sign_count` dataset into one row per speaker, system, and target class by averaging the concept columns across that speaker's utterances.


In [1]:
from pathlib import Path

import pandas as pd


In [2]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'resnet_293' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

INPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_wide.csv'
OUTPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_speaker_level.csv'

# Set True if you want to exclude A12 here.
EXCLUDE_A12 = False

print('INPUT_CSV =', INPUT_CSV)
print('Exists =', INPUT_CSV.exists())
print('OUTPUT_CSV =', OUTPUT_CSV)


INPUT_CSV = /home/SpeakerRec/BioVoice/resnet_293/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_sign_count_wide.csv
Exists = True
OUTPUT_CSV = /home/SpeakerRec/BioVoice/resnet_293/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_sign_count_speaker_level.csv


In [3]:
df = pd.read_csv(INPUT_CSV)

if EXCLUDE_A12:
    df = df[df['system_id'] != 'A12'].copy()

meta_cols = ['utt_id', 'speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']
feature_cols = [c for c in df.columns if c not in meta_cols]

print('Input rows =', len(df))
print('Input utterances =', df['utt_id'].nunique())
print('Input speakers =', df['speaker_id'].nunique())
print('Feature columns =', len(feature_cols))
display(df.head())


Input rows = 12000
Input utterances = 7907
Input speakers = 81
Feature columns = 12


,utt_id,speaker_id,split,source_partition,system_id,target_class,binary_label,long_constant_thick,long_dropping_flat_thick,long_dropping_steep_thick,long_dropping_steep_thin,long_rising_flat_thick,long_rising_steep_thick,long_rising_steep_thin,short_constant_thick,short_dropping_steep_thick,short_dropping_steep_thin,short_rising_steep_thick,short_rising_steep_thin
0,D_0000036016,D_0430,test,dev,A09,bonafide,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D_0000207208,D_0430,test,dev,A09,bonafide,0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
2,D_0000313804,D_0430,test,dev,A09,bonafide,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D_0000437998,D_0430,test,dev,A09,bonafide,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D_0000438019,D_0430,test,dev,A09,bonafide,0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0


In [4]:
group_cols = ['speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']

speaker_df = (
    df.groupby(group_cols, as_index=False)[feature_cols]
    .mean()
)

utt_counts = (
    df.groupby(group_cols, as_index=False)
    .agg(n_utterances=('utt_id', 'nunique'))
)

speaker_df = speaker_df.merge(
    utt_counts,
    on=group_cols,
    how='left',
    validate='one_to_one',
)

speaker_df = speaker_df.sort_values(['source_partition', 'system_id', 'target_class', 'speaker_id']).reset_index(drop=True)

print('Speaker-level rows =', len(speaker_df))
print('Speaker-system-class groups =', len(speaker_df))
print('Mean utterances per group =', speaker_df['n_utterances'].mean())
display(speaker_df.head())


Speaker-level rows = 600
Speaker-system-class groups = 600
Mean utterances per group = 20.0


,speaker_id,split,source_partition,system_id,target_class,binary_label,long_constant_thick,long_dropping_flat_thick,long_dropping_steep_thick,long_dropping_steep_thin,long_rising_flat_thick,long_rising_steep_thick,long_rising_steep_thin,short_constant_thick,short_dropping_steep_thick,short_dropping_steep_thin,short_rising_steep_thick,short_rising_steep_thin,n_utterances
0,D_0430,test,dev,A09,bonafide,0,0.15,0.10,0.10,0.05,0.10,0.15,0.20,0.15,0.15,0.15,0.05,0.0,20
1,D_0461,test,dev,A09,bonafide,0,0.15,0.15,0.20,0.00,0.05,0.10,0.15,0.25,0.05,0.10,0.25,0.2,20
2,D_0546,test,dev,A09,bonafide,0,0.25,0.20,0.20,0.20,0.15,0.20,0.30,0.25,0.30,0.20,0.25,0.2,20
3,D_0956,test,dev,A09,bonafide,0,0.15,0.00,0.15,0.05,0.15,0.10,0.10,0.15,0.10,0.10,0.15,0.1,20
4,D_2327,test,dev,A09,bonafide,0,0.20,0.20,0.25,0.20,0.25,0.15,0.25,0.30,0.20,0.25,0.25,0.3,20


In [5]:
summary_df = pd.DataFrame({
    'rows': [len(speaker_df)],
    'speakers': [speaker_df['speaker_id'].nunique()],
    'systems': [speaker_df['system_id'].nunique()],
    'classes': [speaker_df['target_class'].nunique()],
    'feature_cols': [len(feature_cols)],
    'mean_n_utterances': [speaker_df['n_utterances'].mean()],
    'min_n_utterances': [speaker_df['n_utterances'].min()],
    'max_n_utterances': [speaker_df['n_utterances'].max()],
})
display(summary_df)

display(
    speaker_df.groupby(['system_id', 'target_class'])['speaker_id']
    .nunique()
    .rename('n_speakers')
    .reset_index()
    .sort_values(['system_id', 'target_class'])
)


,rows,speakers,systems,classes,feature_cols,mean_n_utterances,min_n_utterances,max_n_utterances
0,600,81,15,2,12,20.0,20,20


,system_id,target_class,n_speakers
0,A01,bonafide,20
1,A01,spoof,20
2,A02,bonafide,20
3,A02,spoof,20
4,A03,bonafide,20
5,A03,spoof,20
6,A04,bonafide,20
7,A04,spoof,20
8,A05,bonafide,20
9,A05,spoof,20


In [6]:
speaker_df.to_csv(OUTPUT_CSV, index=False)
print('Saved speaker-level sign-count dataset to:', OUTPUT_CSV)


Saved speaker-level sign-count dataset to: /home/SpeakerRec/BioVoice/resnet_293/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_sign_count_speaker_level.csv
